## Converteer conceptlijst in Memorix van turtle naar Excel

In [4]:
import pandas as pd
import json, sys, os
from pathlib import Path

root = '../' # jouw pad naar saa-nexus-scripts
sys.path.insert(0, root)

In [ ]:
from modules import memorix
from modules import saa

settings = saa.readJsonFile('../settings.prod.json') # prod of acc
api = memorix.ApiClient(settings)

FileNotFoundError: [Errno 2] No such file or directory: '../../settings.prod.json'

In [ ]:
# Conceptlijst
vocabulair = 'a4863c0c-d9e5-3902-831a-d0960e381a41' # straten, of kies andere conceptlijst in Memorix

# Bestandspaden
turtle_file = "straten.ttl"       # <-- Zet hier het pad naar jouw Turtle file
excel_file = "straten.xlsx"       # <-- Zet hier de gewenste Excel naam

In [ ]:
response = api.list_concepts(vocabulair)
print(response.text,  file=open(turtle_file, 'w', encoding='utf-8'))

In [ ]:
import rdflib
import pandas as pd
import re

# Laden van RDF/Turtle bestand
g = rdflib.Graph()
g.parse(turtle_file, format="ttl")

# Namespace
SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

rows = []
for s in g.subjects(rdflib.RDF.type, SKOS.Concept):
    s_str = str(s)
    
    match = re.search(r'/vocabularies/concepts/([^/>]+)', s_str)
    uuid = match.group(1) if match else ""

    prefLabel = next((str(lab) for lab in g.objects(s, SKOS.prefLabel)), "")
    exactMatch = next((str(em) for em in g.objects(s, SKOS.exactMatch)), "") # <-- fout: want exactMatch kan nu meer dan 1 waarde hebben
    scopeNote = next((str(sn) for sn in g.objects(s, SKOS.scopeNote)), "")

    rows.append({
        "uuid": uuid,
        "prefLabel": prefLabel,
        "exactMatch": exactMatch,
        "scopeNote": scopeNote
    })

# DataFrame maken en wegschrijven naar Excel
df = pd.DataFrame(rows)
df.to_excel(excel_file, index=False)

print(f"Gereed! Geëxporteerd naar {excel_file}")

Gereed! Geëxporteerd naar straten.xlsx
